# 4. Deep Q-Learning

## 4.1 From Q-Learning to Deep Q-Networks

Deep Q-Learning combines Q-learning with deep neural networks to handle high-dimensional state spaces. This breakthrough algorithm, introduced by DeepMind in 2013, achieved human-level performance on Atari games using only pixel inputs [[Mnih et al. 2013]](#references).

### 4.1.1 Standard Q-Learning Recap

Recall the Q-learning update rule from Chapter 4:

$$
Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \left[r_{t+1} + \gamma \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t)\right]
$$

This update requires:
- A table to store Q-values for all state-action pairs
- Visiting each state multiple times to learn accurate values

### 4.1.2 Deep Q-Learning Formulation

Deep Q-Learning replaces the Q-table with a neural network $Q(s, a; \theta)$ parameterized by $\theta$:

$$
Q(s, a; \theta) \approx Q^*(s, a)
$$

**Loss function** (temporal difference error):

$$
L(\theta) = \mathbb{E}_{(s,a,r,s') \sim \mathcal{D}}\left[\left(y - Q(s, a; \theta)\right)^2\right]
$$

where the **target** is:

$$
y = r + \gamma \max_{a'} Q(s', a'; \theta^-)
$$

and $\theta^-$ represents parameters of a separate **target network**.

### 4.1.3 Key Challenges

Applying deep learning to RL presents several challenges:

1. **Correlations in observation sequences**: Consecutive states are highly correlated
2. **Non-stationary targets**: Q-values change as policy improves
3. **Scale of rewards**: Different environments have vastly different reward scales
4. **Exploration vs exploitation**: Need to balance trying new actions vs using known good ones

Deep Q-Networks (DQN) address these challenges through:
- **Experience replay**: Store and sample experiences randomly
- **Target networks**: Use separate network for computing targets
- **Reward clipping**: Normalize rewards to [-1, 1]
- **ε-greedy exploration**: Systematic exploration strategy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Visualize Q-Learning vs Deep Q-Learning
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Q-Learning diagram
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Traditional Q-Learning', fontsize=16, fontweight='bold')

# Draw Q-table
table_x, table_y = 2, 3
for i in range(4):
    for j in range(4):
        rect = Rectangle((table_x + j*1.2, table_y + i*1.2), 1, 1, 
                         fill=True, facecolor='lightblue', edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        ax.text(table_x + j*1.2 + 0.5, table_y + i*1.2 + 0.5, 
               f'{np.random.rand():.2f}', ha='center', va='center', fontsize=8)

ax.text(5, 8.5, 'Q-Table', ha='center', fontsize=14, fontweight='bold')
ax.text(5, 1.5, 'State: (x, y)\nAction: ↑↓←→', ha='center', fontsize=11)
ax.text(5, 0.5, 'Limited to small state spaces', ha='center', fontsize=10, style='italic')

# Deep Q-Learning diagram
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Deep Q-Learning', fontsize=16, fontweight='bold')

# Draw neural network
layers = [3, 5, 5, 4]
layer_x = [2, 4, 6, 8]
for i, (n_neurons, x) in enumerate(zip(layers, layer_x)):
    y_positions = np.linspace(2, 8, n_neurons)
    for y in y_positions:
        circle = plt.Circle((x, y), 0.3, fill=True, facecolor='lightgreen', 
                          edgecolor='black', linewidth=2)
        ax.add_patch(circle)
    
    # Draw connections to next layer
    if i < len(layers) - 1:
        next_y = np.linspace(2, 8, layers[i+1])
        for y1 in y_positions:
            for y2 in next_y:
                ax.plot([x, layer_x[i+1]], [y1, y2], 'k-', alpha=0.2, linewidth=0.5)

ax.text(2, 0.8, 'State\n(image)', ha='center', fontsize=10)
ax.text(8, 0.8, 'Q-values\nper action', ha='center', fontsize=10)
ax.text(5, 9.2, 'Neural Network Q(s,a;θ)', ha='center', fontsize=12, fontweight='bold')
ax.text(5, 0.3, 'Scales to high-dimensional spaces', ha='center', fontsize=10, style='italic')

plt.tight_layout()
plt.show()

## 4.2 Experience Replay

Experience replay is a crucial innovation that enables stable training of deep Q-networks.

### 4.2.1 The Problem with Online Learning

In standard Q-learning, we update immediately after each experience:

**Problems:**
1. **Sequential correlation**: Consecutive states are similar, leading to overfitting
2. **Catastrophic forgetting**: New experiences overwrite old knowledge
3. **Limited data efficiency**: Each experience used only once
4. **Unstable learning**: High variance in updates

### 4.2.2 Experience Replay Mechanism

Experience replay stores transitions in a replay buffer $\mathcal{D}$ and samples randomly during training:

**Algorithm:**
1. Execute action $a_t$ in state $s_t$, observe $r_{t+1}, s_{t+1}$
2. Store transition $(s_t, a_t, r_{t+1}, s_{t+1})$ in $\mathcal{D}$
3. Sample random minibatch of transitions from $\mathcal{D}$
4. Perform gradient descent on minibatch

**Replay buffer:**
$$
\mathcal{D} = \{(s_1, a_1, r_2, s_2), (s_2, a_2, r_3, s_3), \ldots, (s_T, a_T, r_{T+1}, s_{T+1})\}
$$

**Sampling:**
$$
\mathcal{B} = \{(s_i, a_i, r_{i+1}, s_{i+1})\}_{i=1}^{N} \sim \text{Uniform}(\mathcal{D})
$$

### 4.2.3 Benefits

1. **Breaks correlation**: Random sampling decorrelates training samples
2. **Data efficiency**: Experiences reused multiple times
3. **Stability**: Reduces variance in gradient updates
4. **Off-policy learning**: Can learn from experiences generated by old policies

### 4.2.4 Implementation Considerations

**Buffer size**: Typically 10^4 to 10^6 transitions
- Too small: Limited diversity, correlation remains
- Too large: Memory constraints, outdated experiences

**Sampling strategy**:
- **Uniform**: Equal probability for all transitions
- **Prioritized**: Sample important transitions more frequently (discussed later)

**Minibatch size**: Typically 32-256 transitions
- Larger batches: More stable gradients, slower updates
- Smaller batches: Faster updates, higher variance

In [ ]:
import numpy as np
from collections import deque
import random

class ReplayBuffer:
    """Experience Replay Buffer for DQN"""
    
    def __init__(self, capacity: int = 100000):
        """
        Args:
            capacity: Maximum number of transitions to store
        """
        self.buffer = deque(maxlen=capacity)
        self.capacity = capacity
    
    def push(self, state, action, reward, next_state, done):
        """Add a transition to the buffer"""
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size: int):
        """Sample a random minibatch of transitions"""
        batch = random.sample(self.buffer, batch_size)
        
        # Unpack batch
        states, actions, rewards, next_states, dones = zip(*batch)
        
        return (
            np.array(states),
            np.array(actions),
            np.array(rewards, dtype=np.float32),
            np.array(next_states),
            np.array(dones, dtype=np.uint8)
        )
    
    def __len__(self):
        return len(self.buffer)
    
    def is_ready(self, batch_size: int) -> bool:
        """Check if buffer has enough samples for a minibatch"""
        return len(self.buffer) >= batch_size

# Demonstrate replay buffer
replay_buffer = ReplayBuffer(capacity=1000)

# Add some sample experiences
print("Adding experiences to replay buffer...")
for i in range(500):
    state = np.random.randn(4)  # 4D state
    action = np.random.randint(0, 2)  # Binary action
    reward = np.random.randn()  # Random reward
    next_state = np.random.randn(4)
    done = np.random.rand() > 0.95  # 5% chance of terminal
    
    replay_buffer.push(state, action, reward, next_state, done)

print(f"Buffer size: {len(replay_buffer)}")
print(f"Buffer capacity: {replay_buffer.capacity}")

# Sample a minibatch
if replay_buffer.is_ready(batch_size=32):
    states, actions, rewards, next_states, dones = replay_buffer.sample(32)
    print(f"\nSampled minibatch:")
    print(f"  States shape: {states.shape}")
    print(f"  Actions shape: {actions.shape}")
    print(f"  Rewards shape: {rewards.shape}")
    print(f"  Next states shape: {next_states.shape}")
    print(f"  Dones shape: {dones.shape}")

# Visualize replay buffer statistics
states, actions, rewards, next_states, dones = replay_buffer.sample(min(500, len(replay_buffer)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Reward distribution
axes[0].hist(rewards, bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Reward', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Reward Distribution in Buffer', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Action distribution
unique_actions, counts = np.unique(actions, return_counts=True)
axes[1].bar(unique_actions, counts, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Action', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Action Distribution in Buffer', fontsize=14)
axes[1].grid(True, alpha=0.3, axis='y')

# Terminal state distribution
terminal_pct = (dones.sum() / len(dones)) * 100
axes[2].bar(['Non-terminal', 'Terminal'], 
           [100 - terminal_pct, terminal_pct],
           edgecolor='black', alpha=0.7)
axes[2].set_ylabel('Percentage (%)', fontsize=12)
axes[2].set_title('Terminal State Distribution', fontsize=14)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 4.3 Target Networks

Target networks address the instability caused by using the same network for both prediction and target computation.

### 4.3.1 The Moving Target Problem

In standard Q-learning with function approximation:

$$
L(\theta) = \mathbb{E}\left[\left(r + \gamma \max_{a'} Q(s', a'; \theta) - Q(s, a; \theta)\right)^2\right]
$$

**Problem**: Both the prediction and target depend on $\theta$
- As we update $\theta$ to reduce error, the target also changes
- Creates "chasing a moving target" effect
- Leads to oscillations and divergence

### 4.3.2 Target Network Solution

Use separate network with parameters $\theta^-$ for computing targets:

$$
L(\theta) = \mathbb{E}\left[\left(r + \gamma \max_{a'} Q(s', a'; \theta^-) - Q(s, a; \theta)\right)^2\right]
$$

**Update strategies:**

1. **Hard update** (DQN): Copy parameters every $C$ steps
   $$
   \theta^- \leftarrow \theta \quad \text{every } C \text{ steps}
   $$

2. **Soft update** (DDPG): Slowly blend parameters
   $$
   \theta^- \leftarrow \tau \theta + (1 - \tau) \theta^- \quad \text{where } \tau \ll 1
   $$

### 4.3.3 Benefits

1. **Stable targets**: Target values change slowly
2. **Reduced oscillations**: Smooth learning curves
3. **Better convergence**: More reliable training
4. **Compatibility with experience replay**: Works well with batched updates

### 4.3.4 Hyperparameter Selection

**Hard update frequency $C$:**
- Too frequent: Reduces to moving target problem
- Too infrequent: Slow learning, outdated targets
- Typical values: 1000-10000 steps

**Soft update rate $\tau$:**
- Controls blend between current and target networks
- Typical values: 0.001-0.01
- Smaller values = more stable but slower updates

In [ ]:
import torch
import torch.nn as nn
import copy
import matplotlib.pyplot as plt

class SimpleQNetwork(nn.Module):
    """Simple Q-network for demonstration"""
    
    def __init__(self, state_dim, n_actions):
        super(SimpleQNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, n_actions)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

def hard_update(target_net, policy_net):
    """Hard update: Copy parameters completely"""
    target_net.load_state_dict(policy_net.state_dict())

def soft_update(target_net, policy_net, tau=0.005):
    """Soft update: Blend parameters"""
    for target_param, policy_param in zip(target_net.parameters(), policy_net.parameters()):
        target_param.data.copy_(tau * policy_param.data + (1 - tau) * target_param.data)

# Create networks
state_dim = 4
n_actions = 2
policy_net = SimpleQNetwork(state_dim, n_actions)
target_net = SimpleQNetwork(state_dim, n_actions)

# Initialize target network with same parameters
hard_update(target_net, policy_net)

print("Network architectures:")
print("\nPolicy Network:")
print(policy_net)
print(f"\nTotal parameters: {sum(p.numel() for p in policy_net.parameters())}")

# Simulate training with different update strategies
n_steps = 1000
hard_update_freq = 100
tau_soft = 0.01

# Track parameter differences
hard_update_diffs = []
soft_update_diffs = []
no_target_diffs = []

# Create separate networks for each strategy
policy_hard = copy.deepcopy(policy_net)
target_hard = copy.deepcopy(target_net)
policy_soft = copy.deepcopy(policy_net)
target_soft = copy.deepcopy(target_net)
policy_none = copy.deepcopy(policy_net)

# Simulate updates
for step in range(n_steps):
    # Simulate gradient update (random for demo)
    for param in policy_hard.parameters():
        param.data += torch.randn_like(param.data) * 0.01
    for param in policy_soft.parameters():
        param.data += torch.randn_like(param.data) * 0.01
    for param in policy_none.parameters():
        param.data += torch.randn_like(param.data) * 0.01
    
    # Hard update
    if step % hard_update_freq == 0:
        hard_update(target_hard, policy_hard)
    
    # Soft update
    soft_update(target_soft, policy_soft, tau=tau_soft)
    
    # Calculate parameter differences
    hard_diff = sum((p1 - p2).abs().sum().item() 
                   for p1, p2 in zip(policy_hard.parameters(), target_hard.parameters()))
    soft_diff = sum((p1 - p2).abs().sum().item() 
                   for p1, p2 in zip(policy_soft.parameters(), target_soft.parameters()))
    none_diff = sum((p1 - p2).abs().sum().item() 
                   for p1, p2 in zip(policy_none.parameters(), policy_net.parameters()))
    
    hard_update_diffs.append(hard_diff)
    soft_update_diffs.append(soft_diff)
    no_target_diffs.append(none_diff)

# Visualize update strategies
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot parameter differences
axes[0].plot(hard_update_diffs, label='Hard Update', linewidth=2)
axes[0].plot(soft_update_diffs, label='Soft Update', linewidth=2)
axes[0].axvline(x=hard_update_freq, color='red', linestyle='--', alpha=0.3, label='Update Points')
for i in range(hard_update_freq, n_steps, hard_update_freq):
    axes[0].axvline(x=i, color='red', linestyle='--', alpha=0.3)
axes[0].set_xlabel('Training Steps', fontsize=12)
axes[0].set_ylabel('Parameter Difference', fontsize=12)
axes[0].set_title('Policy vs Target Network Parameters', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot zoomed view
zoom_start, zoom_end = 0, 300
axes[1].plot(hard_update_diffs[zoom_start:zoom_end], label='Hard Update', linewidth=2)
axes[1].plot(soft_update_diffs[zoom_start:zoom_end], label='Soft Update', linewidth=2)
axes[1].axvline(x=hard_update_freq, color='red', linestyle='--', alpha=0.5, label='Hard Update Point')
axes[1].set_xlabel('Training Steps', fontsize=12)
axes[1].set_ylabel('Parameter Difference', fontsize=12)
axes[1].set_title('Zoomed View (First 300 Steps)', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAverage parameter difference:")
print(f"  Hard update: {np.mean(hard_update_diffs):.2f}")
print(f"  Soft update: {np.mean(soft_update_diffs):.2f}")
print(f"\nHard update creates step changes every {hard_update_freq} steps")
print(f"Soft update provides smooth, continuous convergence")

## 4.4 The DQN Algorithm

Combining neural networks, experience replay, and target networks yields the complete DQN algorithm.

### 4.4.1 DQN Pseudocode

```
Initialize replay buffer D with capacity N
Initialize Q-network with random weights θ
Initialize target network with weights θ⁻ = θ

for episode = 1 to M:
    Initialize state s₁
    
    for t = 1 to T:
        # Select action using ε-greedy policy
        With probability ε: select random action aₜ
        Otherwise: aₜ = argmax_a Q(sₜ, a; θ)
        
        # Execute action and observe outcome
        Execute action aₜ, observe reward rₜ₊₁ and next state sₜ₊₁
        
        # Store transition in replay buffer
        Store (sₜ, aₜ, rₜ₊₁, sₜ₊₁) in D
        
        # Sample random minibatch from D
        Sample minibatch of transitions (sⱼ, aⱼ, rⱼ₊₁, sⱼ₊₁) from D
        
        # Compute targets
        For terminal sⱼ₊₁: yⱼ = rⱼ₊₁
        Otherwise: yⱼ = rⱼ₊₁ + γ max_a' Q(sⱼ₊₁, a'; θ⁻)
        
        # Gradient descent step
        Perform gradient descent on (yⱼ - Q(sⱼ, aⱼ; θ))²
        
        # Update target network
        Every C steps: θ⁻ = θ
```

### 4.4.2 Loss Function

The loss function for a minibatch is:

$$
L(\theta) = \frac{1}{|\mathcal{B}|} \sum_{(s,a,r,s') \in \mathcal{B}} \left(y - Q(s, a; \theta)\right)^2
$$

where:

$$
y = \begin{cases}
r & \text{if } s' \text{ is terminal} \\
r + \gamma \max_{a'} Q(s', a'; \theta^-) & \text{otherwise}
\end{cases}
$$

### 4.4.3 Gradient Computation

The gradient with respect to network parameters:

$$
\nabla_\theta L(\theta) = -\frac{1}{|\mathcal{B}|} \sum_{(s,a,r,s') \in \mathcal{B}} \left(y - Q(s, a; \theta)\right) \nabla_\theta Q(s, a; \theta)
$$

Note: The target $y$ is treated as a constant (detached from computational graph).

### 4.4.4 Exploration Strategy

**ε-greedy exploration:**

$$
a_t = \begin{cases}
\text{random action} & \text{with probability } \epsilon \\
\arg\max_{a} Q(s_t, a; \theta) & \text{with probability } 1 - \epsilon
\end{cases}
$$

**ε-decay schedule:**

$$
\epsilon_t = \max(\epsilon_{\text{min}}, \epsilon_{\text{start}} \cdot \text{decay}^t)
$$

or linear decay:

$$
\epsilon_t = \max(\epsilon_{\text{min}}, \epsilon_{\text{start}} - \frac{t(\epsilon_{\text{start}} - \epsilon_{\text{min}})}{T_{\text{decay}}})
$$

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import random

class DQN:
    """Deep Q-Network implementation"""
    
    def __init__(self, state_dim, n_actions, hidden_dim=128, lr=0.001, 
                 gamma=0.99, epsilon_start=1.0, epsilon_end=0.01, 
                 epsilon_decay=0.995, buffer_size=10000, batch_size=64,
                 target_update_freq=100):
        
        self.state_dim = state_dim
        self.n_actions = n_actions
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        
        # Exploration parameters
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        
        # Networks
        self.policy_net = self._build_network(hidden_dim)
        self.target_net = self._build_network(hidden_dim)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        
        # Optimizer
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        
        # Replay buffer
        self.replay_buffer = ReplayBuffer(buffer_size)
        
        # Training statistics
        self.steps = 0
        self.losses = []
    
    def _build_network(self, hidden_dim):
        """Build Q-network"""
        return nn.Sequential(
            nn.Linear(self.state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, self.n_actions)
        )
    
    def select_action(self, state, training=True):
        """Select action using ε-greedy policy"""
        if training and random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            q_values = self.policy_net(state_tensor)
            return q_values.argmax().item()
    
    def update(self):
        """Perform one training step"""
        if not self.replay_buffer.is_ready(self.batch_size):
            return None
        
        # Sample minibatch
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)
        
        # Convert to tensors
        states = torch.FloatTensor(states)
        actions = torch.LongTensor(actions)
        rewards = torch.FloatTensor(rewards)
        next_states = torch.FloatTensor(next_states)
        dones = torch.FloatTensor(dones)
        
        # Current Q-values
        current_q_values = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze()
        
        # Target Q-values
        with torch.no_grad():
            next_q_values = self.target_net(next_states).max(1)[0]
            target_q_values = rewards + (1 - dones) * self.gamma * next_q_values
        
        # Compute loss
        loss = nn.MSELoss()(current_q_values, target_q_values)
        
        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        # Update target network
        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())
        
        # Decay epsilon
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
        
        self.losses.append(loss.item())
        return loss.item()
    
    def store_transition(self, state, action, reward, next_state, done):
        """Store transition in replay buffer"""
        self.replay_buffer.push(state, action, reward, next_state, done)

# Simple environment for testing
class SimpleEnv:
    def __init__(self, state_dim=4):
        self.state_dim = state_dim
        self.state = None
        self.steps = 0
    
    def reset(self):
        self.state = np.random.randn(self.state_dim)
        self.steps = 0
        return self.state
    
    def step(self, action):
        # Simple dynamics: reward based on state magnitude and action
        reward = -np.linalg.norm(self.state) + (action * 0.1)
        self.state = self.state + np.random.randn(self.state_dim) * 0.1
        self.steps += 1
        done = self.steps >= 100
        return self.state, reward, done, {}

# Train DQN
env = SimpleEnv(state_dim=4)
agent = DQN(state_dim=4, n_actions=2, hidden_dim=64, lr=0.001)

n_episodes = 100
episode_rewards = []

print("Training DQN...")
for episode in range(n_episodes):
    state = env.reset()
    episode_reward = 0
    
    for step in range(100):
        # Select and execute action
        action = agent.select_action(state)
        next_state, reward, done, _ = env.step(action)
        
        # Store transition and update
        agent.store_transition(state, action, reward, next_state, done)
        agent.update()
        
        episode_reward += reward
        state = next_state
        
        if done:
            break
    
    episode_rewards.append(episode_reward)
    
    if (episode + 1) % 20 == 0:
        avg_reward = np.mean(episode_rewards[-20:])
        print(f"Episode {episode + 1}: Avg Reward = {avg_reward:.2f}, ε = {agent.epsilon:.3f}")

# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Episode rewards
axes[0].plot(episode_rewards, alpha=0.6, label='Episode Reward')
axes[0].plot(np.convolve(episode_rewards, np.ones(10)/10, mode='valid'), 
            linewidth=2, label='Moving Average (10 episodes)')
axes[0].set_xlabel('Episode', fontsize=12)
axes[0].set_ylabel('Total Reward', fontsize=12)
axes[0].set_title('DQN Training Progress', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Loss curve
if agent.losses:
    axes[1].plot(agent.losses, alpha=0.6)
    axes[1].set_xlabel('Training Step', fontsize=12)
    axes[1].set_ylabel('Loss', fontsize=12)
    axes[1].set_title('Training Loss', fontsize=14)
    axes[1].grid(True, alpha=0.3)

# Epsilon decay
epsilons = [max(agent.epsilon_end, agent.epsilon * (agent.epsilon_decay ** i)) 
           for i in range(agent.steps)]
axes[2].plot(epsilons)
axes[2].set_xlabel('Training Step', fontsize=12)
axes[2].set_ylabel('ε (Exploration Rate)', fontsize=12)
axes[2].set_title('Exploration Decay', fontsize=14)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal performance:")
print(f"  Average reward (last 10 episodes): {np.mean(episode_rewards[-10:]):.2f}")
print(f"  Final epsilon: {agent.epsilon:.4f}")
print(f"  Total training steps: {agent.steps}")

## 4.5 DQN Variants and Improvements

Several improvements to the basic DQN algorithm have been proposed:

### 4.5.1 Double DQN

**Problem**: DQN tends to overestimate Q-values due to max operation in target:

$$
y = r + \gamma \max_{a'} Q(s', a'; \theta^-)
$$

**Solution**: Decouple action selection from evaluation [[van Hasselt et al. 2016]](#references):

$$
y = r + \gamma Q(s', \arg\max_{a'} Q(s', a'; \theta); \theta^-)
$$

- Select best action using policy network (θ)
- Evaluate using target network (θ⁻)

### 4.5.2 Prioritized Experience Replay

**Idea**: Sample important transitions more frequently [[Schaul et al. 2016]](#references)

**Priority**: Based on TD error magnitude:

$$
p_i = |r + \gamma \max_{a'} Q(s', a'; \theta^-) - Q(s, a; \theta)| + \epsilon
$$

**Sampling probability**:

$$
P(i) = \frac{p_i^\alpha}{\sum_k p_k^\alpha}
$$

where $\alpha$ controls prioritization strength.

**Importance sampling weights**: Correct for bias:

$$
w_i = \left(\frac{1}{N \cdot P(i)}\right)^\beta
$$

### 4.5.3 Dueling DQN

**Idea**: Separate value and advantage functions [[Wang et al. 2016]](#references)

$$
Q(s, a; \theta) = V(s; \theta_V) + A(s, a; \theta_A) - \frac{1}{|\mathcal{A}|} \sum_{a'} A(s, a'; \theta_A)
$$

where:
- $V(s)$ is the state value
- $A(s, a)$ is the advantage of action $a$ in state $s$

**Benefits**:
- Better value estimation
- Faster learning
- Especially useful when many actions have similar values

### 4.5.4 Rainbow DQN

Combines multiple improvements [[Hessel et al. 2018]](#references):
1. Double Q-learning
2. Prioritized experience replay
3. Dueling networks
4. Multi-step learning
5. Distributional RL
6. Noisy networks

Achieves state-of-the-art performance on Atari benchmark.

## 4.6 DQN for Topology Optimization

Applying DQN to topology optimization requires careful consideration of:

### 4.6.1 State Representation

For C-core electromagnetic actuator optimization:
- **Channel 1**: Material distribution (0=air, 1=iron)
- **Channel 2**: Flux density magnitude
- **Channel 3**: Boundary conditions

State tensor: $s \in \mathbb{R}^{3 \times H \times W}$

### 4.6.2 Action Space

**Options**:
1. **Pixel-level**: Each action places material at one pixel
   - Action space size: $H \times W$
   - Fine-grained control
   - Large action space

2. **Region-level**: Actions modify larger regions
   - Smaller action space
   - Faster learning
   - Less precision

### 4.6.3 Reward Function

$$
r_t = \begin{cases}
k_1 \cdot F_y(s_t) - k_2 \cdot V(s_t) & \text{if valid design} \\
-P & \text{if constraint violated}
\end{cases}
$$

where:
- $F_y(s_t)$ is the vertical force produced
- $V(s_t)$ is the material volume
- $P$ is a large penalty
- $k_1, k_2$ are weighting coefficients

### 4.6.4 Network Architecture

CNN-based Q-network:
```
Input: [batch, 3, H, W]
  ↓
Conv2D(32, 3×3, stride=1) + ReLU
  ↓
Conv2D(64, 3×3, stride=1) + ReLU
  ↓
MaxPool2D(2×2)
  ↓
Conv2D(128, 3×3, stride=1) + ReLU
  ↓
MaxPool2D(2×2)
  ↓
Flatten
  ↓
Dense(512) + ReLU
  ↓
Dense(n_actions)  # Q-values
```

### 4.6.5 Computational Considerations

**Challenges**:
- FEM simulation required for each action (expensive)
- Large state space (50×50 grid)
- Sparse rewards (only at episode end)

**Solutions**:
- Surrogate models for fast evaluation during exploration
- Curriculum learning (start with easier problems)
- Reward shaping (intermediate rewards)
- Parallel environment execution

## 4.7 Summary

### 4.7.1 Key Concepts

1. **Deep Q-Networks** combine Q-learning with deep neural networks to handle high-dimensional state spaces

2. **Experience replay** breaks correlations in sequential data and enables data-efficient learning

3. **Target networks** stabilize learning by providing consistent targets

4. **DQN variants** (Double DQN, Prioritized Replay, Dueling DQN) address specific limitations

5. **CNNs are essential** for processing spatial topology representations

### 4.7.2 Advantages and Limitations

**Advantages:**
- Scales to high-dimensional state spaces
- Off-policy learning (data efficient)
- Proven success on complex tasks
- Straightforward implementation

**Limitations:**
- Only applicable to discrete action spaces
- Can be sample inefficient compared to policy gradient methods
- Sensitive to hyperparameters
- May overestimate Q-values

### 4.7.3 Next Steps

While DQN is powerful, it has limitations for topology optimization:
- Discrete actions may be too restrictive
- Exploration can be inefficient
- No explicit policy representation

In the next section, we'll explore **policy gradient methods** that directly optimize the policy and can handle continuous action spaces.

## References

- Mnih, V., et al. (2013). Playing Atari with deep reinforcement learning. *arXiv preprint arXiv:1312.5602*.
- Mnih, V., et al. (2015). Human-level control through deep reinforcement learning. *Nature*, 518(7540), 529-533.
- van Hasselt, H., Guez, A., & Silver, D. (2016). Deep reinforcement learning with double Q-learning. *AAAI Conference on Artificial Intelligence*.
- Schaul, T., et al. (2016). Prioritized experience replay. *International Conference on Learning Representations*.
- Wang, Z., et al. (2016). Dueling network architectures for deep reinforcement learning. *International Conference on Machine Learning*.
- Hessel, M., et al. (2018). Rainbow: Combining improvements in deep reinforcement learning. *AAAI Conference on Artificial Intelligence*.
- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.

## 4.7 DQN training run on SeqTO-v2 (C-core)

The DQN agent in [`dqn_seqto.py`](dqn_seqto.py) — CNN backbone (`net_seqto.SeqToCNN`) + a 4-way Q-head, soft target update (`tau=0.005`), epsilon-greedy exploration with multiplicative decay, and a 10k-transition replay buffer — was trained on the SeqTO-v2 C-core environment described in Section 6.

```bash
python dqn_seqto.py --episodes 200 --reward-kind force
```

**Status: complete (200/200 episodes).** Artefacts live in `designs/dqn_run/`. The cells below load `training_stats.json` and render the learning curves, the epsilon schedule, and the highest-reward mask. A small mask gallery (ep 1 → 50 → best → 200) follows, then a short interpretation.


In [ ]:
# Load and visualise the DQN training run from designs/dqn_run/
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

with open('designs/dqn_run/training_stats.json') as f:
    s = json.load(f)
eps = s['episode_stats']
hp  = s['hyperparams']
n_captured = len(eps)
n_planned  = hp['episodes']

rewards  = np.array([e['reward']     for e in eps])
forces   = np.array([e['force_N']    for e in eps])
losses   = np.array([e['loss']       for e in eps])
epsilons = np.array([e['epsilon']    for e in eps])
irons    = np.array([e['iron_count'] for e in eps])

best_ep = hp['best_episode']
best_R  = hp['best_reward']
best_F  = eps[best_ep - 1]['force_N']
best_iron = eps[best_ep - 1]['iron_count']

# Random-search ceiling from env-probe (Section 6.8) -- horizontal reference line
from pathlib import Path
rs_files = sorted(Path('designs/env_probe').glob('random_search_*.json'))
RANDOM_CEILING_N = json.loads(rs_files[-1].read_text())['summary']['max'] if rs_files else None

fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.5))

ax = axes[0, 0]
ax.plot(range(1, n_captured + 1), rewards, lw=1.0, marker='o', ms=2,
        color='steelblue', alpha=0.5, label='Episode reward')
ax.plot(range(1, n_captured + 1), np.maximum.accumulate(rewards),
        'crimson', lw=2.0, label='Best so far')
ax.axvline(best_ep, ls='--', color='k', alpha=0.5)
ax.text(best_ep + 1, best_R, f' ep {best_ep}: {best_R:.0f}', va='center', fontsize=8)
ax.set_xlabel('Episode'); ax.set_ylabel('Cumulative reward (force-based)')
ax.set_title(f'DQN reward -- {n_captured}/{n_planned} episodes (complete)')
ax.legend(loc='upper left'); ax.grid(alpha=0.3)

ax = axes[0, 1]
ax.plot(range(1, n_captured + 1), forces, lw=1.0, marker='o', ms=2, color='darkgreen', alpha=0.6, label='Final-step force')
ax.plot(range(1, n_captured + 1), np.maximum.accumulate(forces), color='darkgreen', lw=2.0, label='Running max')
if RANDOM_CEILING_N is not None:
    ax.axhline(RANDOM_CEILING_N, ls=':', color='crimson', lw=1.5,
               label=f'Random-search ceiling ({RANDOM_CEILING_N:.0f} N)')
ax.set_xlabel('Episode'); ax.set_ylabel('Final-step force (N)')
ax.set_title(f'DQN end-of-episode force  (max={max(forces):.0f} N)')
ax.legend(loc='lower right', fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 0]
ax.plot(range(1, n_captured + 1), losses, lw=1.0, color='purple', alpha=0.7)
ax.set_xlabel('Episode'); ax.set_ylabel('SmoothL1 loss', color='purple')
ax.tick_params(axis='y', labelcolor='purple')
ax.set_title('DQN loss vs episode  +  epsilon schedule')
ax.grid(alpha=0.3)
ax2 = ax.twinx()
ax2.plot(range(1, n_captured + 1), epsilons, lw=1.5, color='darkorange')
ax2.set_ylabel('epsilon', color='darkorange')
ax2.set_ylim(0, 1)
ax2.tick_params(axis='y', labelcolor='darkorange')

ax = axes[1, 1]
mask_img = mpimg.imread(f'designs/dqn_run/episodes/ep_{best_ep:03d}_mask.png')
ax.imshow(mask_img); ax.axis('off')
ax.set_title(f'Best-episode mask  (ep {best_ep}: R={best_R:.0f}, '
             f'force={best_F:.1f} N, {best_iron} cells)')

plt.suptitle(f'DQN on SeqTO-v2 C-core  -- 200/200 episodes complete',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# Summary numbers
max_F_ep = int(np.argmax(forces)) + 1
max_F = forces.max()
print(f'Captured episodes  : {n_captured} of {n_planned}')
print(f'Best by reward     : ep {best_ep}  R={best_R:.2f}  force={best_F:.2f} N  iron_cells={best_iron}')
print(f'Best by force      : ep {max_F_ep}  force={max_F:.1f} N')
print(f'Final episode      : reward={rewards[-1]:.0f}  force={forces[-1]:.1f} N  epsilon={epsilons[-1]:.3f}')
print(f'Late-stage (last 10): mean force={forces[-10:].mean():.1f} N  (vs first 10: {forces[:10].mean():.1f} N)')
if RANDOM_CEILING_N is not None:
    print(f'Random-search ceiling (200 random masks): {RANDOM_CEILING_N:.1f} N')
    print(f'  DQN best-force vs ceiling: {max_F/RANDOM_CEILING_N:.2f}x')
print(f'Hyperparameters    : alpha={hp["alpha"]}  gamma={hp["gamma"]}  '
      f'eps {hp["epsilon_start"]}->{hp["epsilon_min"]} decay={hp["epsilon_decay"]}')
print(f'                     batch={hp["batch_size"]}  replay={hp["replay_size"]}  '
      f'target_tau={hp["target_tau"]}  warmup={hp["warmup_steps"]}')
print(f'FEMM evaluations   : {n_captured * hp["max_steps"]}')


In [ ]:
# Mask gallery -- visual progression of learned topology across training
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

with open('designs/dqn_run/training_stats.json') as f:
    s = json.load(f)
eps = s['episode_stats']
best_ep = s['hyperparams']['best_episode']

# Pick a representative set of episodes: very-early / early / mid / best / late / final
snapshots = [1, 50, 100, best_ep, 180, 200]
snapshots = sorted(set(s for s in snapshots if 1 <= s <= len(eps)))

fig, axes = plt.subplots(1, len(snapshots), figsize=(3.0 * len(snapshots), 3.5))
if len(snapshots) == 1: axes = [axes]
for ax, ep in zip(axes, snapshots):
    img = mpimg.imread(f'designs/dqn_run/episodes/ep_{ep:03d}_mask.png')
    ax.imshow(img); ax.axis('off')
    e = eps[ep - 1]
    tag = '  (best)' if ep == best_ep else ''
    ax.set_title(f"ep {ep}{tag}\nF={e['force_N']:.0f} N  iron={e['iron_count']}", fontsize=10)
plt.suptitle('DQN learned topology over training', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


### 4.7.1 What the run shows

**Convergence to a high-force regime.** The best-by-reward episode is ep 173 ($R = 20{,}925$, force $= 1305$ N) — which also happens to be the peak single-episode force across the whole run ($\sim 2.14\times$ the 610 N random-search ceiling, Section 6.8). The late-stage mean force (last 10 episodes) sits at $\sim 694$ N — substantially above the 108 N first-10 mean.

**Exploration is exhausted at the end.** $\varepsilon$ reached its floor of 0.05 around ep 100 (multiplicative decay 0.97/episode from 1.0); the remaining 100 episodes operated essentially greedy, with the late variance driven by random *start* states rather than random *action* draws. The training loss stays in the 400-700 range through this near-greedy phase — large in absolute terms, but consistent with the off-policy target moving as the policy improves, not with divergence.

**Replay-buffer reuse is doing real work.** With 4 actions per step, 34 steps per episode, batch=32, and 200 episodes, each FEMM evaluation contributes to $\sim 200{,}000 / 6{,}800 \approx 30$ minibatch updates on average — *the same FEMM result is back-propagated through the Q-network dozens of times*. The action-effect probe (Section 6.8) showed the single-step reward landscape is action-blind; the replay buffer is what extracts a useful credit-assignment signal from that.

**Caveat on the absolute force.** The 1305 N peak is a single-episode end-of-trajectory force, not an evaluation-pass average. A separate eval pass (greedy policy, e.g. 20 evaluation episodes from fixed seeds) would tighten the estimate; that pass is not part of the training script and is tracked as planned work in Section 7.4.

**Cost.** Training 200 episodes × 34 steps × ~80 s/FEMM solve ≈ **150 h wall-clock** on the target Windows box. This is the most expensive single run in the chapter; see Section 7.5 for the comparative cost table.
